In [ ]:
from __future__ import annotations

import json
import math
import pathlib
from collections import defaultdict

import pandas as pd
from kebab.utils.dataset.wikidata.wikidata_utils import ResolvedWikidataEntity

In [ ]:
dataset_path = (
    pathlib.Path.home()
    / "OneDrive - Microsoft"
    / "Benchmark"
    / "Datasets"
    / "REBEL"
    / "Linking Dataset Base"
    / "rebel_linking_pairs_dataset.jsonl"
)

ground_truth_path = (
    pathlib.Path.home()
    / "OneDrive - Microsoft"
    / "Benchmark"
    / "Datasets"
    / "REBEL"
    / "Linking Dataset Base"
    / "rebel_linking_pairs_ground_truth.jsonl"
)

Load the fragments

In [ ]:
# load the ground truth
with open(ground_truth_path, encoding="utf-8") as f:
    labels = [json.loads(line) for line in f]

# load fragments
fragments = {}
pairs = []


def add_fragment(fragment: ResolvedWikidataEntity) -> ResolvedWikidataEntity:
    """Add the fragment to the set of known fragments."""
    if fragment.metadata["fragment_id"] not in fragments:
        fragments[fragment.metadata["fragment_id"]] = fragment
        fragment.evidence_map = None
        fragment.source_ids = None

    return fragments[fragment.metadata["fragment_id"]]


with open(dataset_path, encoding="utf-8") as f:
    for line in f:
        d = json.loads(line)
        left = add_fragment(ResolvedWikidataEntity.from_dict(d["left"]))
        right = add_fragment(ResolvedWikidataEntity.from_dict(d["right"]))

        pairs.append((left, right))

print(f"Loaded {len(pairs):,d} pairs containing {len(fragments):,d} distinct fragments")

Example fragment

In [ ]:
pairs[0][0]

In [ ]:
data = defaultdict(list)

for pair, label in zip(pairs, labels):
    left, right = pair
    data["same_entity"].append(label)
    data["same_type"].append(left.wikidata_type == right.wikidata_type)
    data["name_overlap"].append(len(set(left.names).intersection(right.names)) > 0)
    data["lowercase_name_overlap"].append(
        len({n.lower() for n in left.names}.intersection({n.lower() for n in right.names})) > 0
    )

df = pd.DataFrame.from_dict(data)

Linking by name
---

In [ ]:
key = "lowercase_name_overlap"
total = df.shape[0]
positives = df[(df["same_entity"])].shape[0]
negatives = df[(~df["same_entity"])].shape[0]

tp = df[(df["same_entity"]) & (df[key])].shape[0]
fp = df[(~df["same_entity"]) & (df[key])].shape[0]
tn = df[(~df["same_entity"]) & (~df[key])].shape[0]
fn = df[(df["same_entity"]) & (~df[key])].shape[0]

assert tp + fp + tn + fn == total

precision = tp / (tp + fp)
recall = tp / (tp + fn)
tpr = precision
tnr = tn / (tn + fn)

print("Ground truth:")
print(f"Positives = {positives:,d} ({positives / total:.1%})")
print(f"Negatives = {negatives:,d} ({negatives / total:.1%})")
print()
print("Linking by name:")
print(f"True positives = {tp:,d} ({tp / total:.1%})")
print(f"False positives = {fp:,d} ({fp / total:.1%})")
print(f"True negatives = {tn:,d} ({tn / total:.1%})")
print(f"False negatives = {fn:,d} ({fn / total:.1%})")
print()
print(f"Precision = {precision:.4f}")
print(f"Recall = {recall:.4f}")
print()
print(f"P(positive | predict positive) = {tpr:.4f}")
print(f"P(negative | predict negative) = {tnr:.4f}")

In [ ]:
log_prob = tp * math.log(tpr) + fp * math.log(1 - tpr) + tn * math.log(tnr) + fn * math.log(1 - tnr)
print(f"LogProb = {log_prob:.4f}")